In [135]:
import numpy as np
import pandas as pd
import math
from pydrake.all import LinearQuadraticRegulator, DiscreteTimeLinearQuadraticRegulator

from pydrake.all import (
    DiagramBuilder,
    LinearSystem,
    FiniteHorizonLinearQuadraticRegulator,
    FiniteHorizonLinearQuadraticRegulatorOptions,
    Simulator,
)

m = 1.535
l = 0.25
I_xxt = 0.029138900000000002
I_yyt = 0.030227416
I_zzt = 0.056327416000000005
I_zzp = 0.000273104
k_tau = 0.06
gamma = 0.007811651

p_eq = 0
q_eq = 2.25361451700798
r_eq = 11.694423666728527
w1_eq = 600.4122857965222
w2_eq = 768.5277258195483
w3_eq = 600.4122857965222
w4_eq = 0.0
nx_eq = 0.0
ny_eq = 0.1892268837006408
nz_eq = 0.9819333920816344


In [136]:

# Calculate a_const,b_const,c_const,d_const
a_const = ((I_xxt - I_zzt) * r_eq / I_xxt) + I_zzp * (
    w1_eq + w2_eq + w3_eq + w4_eq
) / I_xxt
b_const = (I_xxt - I_zzt) * q_eq / I_xxt
c_const = (I_zzt - I_xxt) * p_eq / I_xxt
d_const = -gamma / I_zzt

# Define matrices A, B, Q, R
# Define the matrix A without deltat
A = np.array(
    [
        [1, a_const, b_const, 0, 0],
        [-a_const, 1, c_const, 0, 0],
        [0, 0, d_const + 1, 0, 0],
        [0, -nz_eq, ny_eq, 1, r_eq],
        [nz_eq, 0, -nx_eq, -r_eq, 1],
    ]
)

B = np.array(
    [
        [0, l / I_xxt, 0],
        [-l / I_xxt, 0, l / I_xxt],
        [k_tau / I_zzt, -k_tau / I_zzt, k_tau / I_zzt],
        [0, 0, 0],
        [0, 0, 0],
    ]
)

Q = np.array(
    [
        [10, 0, 0, 0, 0],
        [0, 10, 0, 0, 0],
        [0, 0, 10, 0, 0],
        [0, 0, 0, 400, 0],
        [0, 0, 0, 0, 400],
        # [10, 0, 0, 0, 0],
        # [0, 10, 0, 0, 0],
        # [0, 0, 10, 0, 0],
        # [0, 0, 0, 575, 0],
        # [0, 0, 0, 0, 575],
    ]
)


R = np.array(
    [
        [1, 0, 0],
        [0, 1, 0],
        [0, 0, 1],
    ]
)


In [137]:

# Calculate the LQR gain
result = LinearQuadraticRegulator(A, B, Q, R)


# Print the result[0] in C++ style initialization format
result_cpp_format = (
    "K = {\n"
    + ",\n".join(
        ["  {" + ", ".join(f"{x:.8e}" for x in row) + "}" for row in result[0]]
    )
    + "\n};"
)

print(result_cpp_format)


def compute_control_input(x):
    """
    Compute the control input vector u given the state vector x.

    :param x: State vector
    :return: Control input vector u
    """
    # Calculate the control input u = -Kx
    y = [float(x[0]), float(x[1]) - 2.53, float(x[2]), float(x[3]) - 0.2855]
    u = -K @ y
    return u



K = result[0]

eigenvalues, eigenvectors = np.linalg.eig(A - B @ K)

print("Eigenvalues of A - B @ K:", eigenvalues)

K = {
  {1.10533273e-01, -2.70560552e+00, 2.39746117e+00, 1.79598554e+01, 2.61505621e+00},
  {4.03213779e+00, 1.67276046e-01, -1.59862246e-01, -3.81114233e+00, 2.36141012e+01},
  {4.86933401e-01, 2.74745356e+00, 2.73452373e+00, -1.63795701e+01, 1.08682548e+00}
};
Eigenvalues of A - B @ K: [-32.29705684 +4.92942424j -32.29705684 -4.92942424j
  -6.34829472+11.87857859j  -6.34829472-11.87857859j
  -4.86401928 +0.j        ]


In [138]:
damping_const = math.sqrt(7.63805378**2/(7.63805378**2 + 11.97289945**2))
natural_freq = 7.63805378 / damping_const
natural_freq_hz = natural_freq / math.pi
damping_const, natural_freq, natural_freq_hz

(0.5378241850923101, 14.201767030407963, 4.520562847057869)

In [139]:
damping_const = math.sqrt(32.02681072**2/(32.02681072**2 + 4.76411854**2))
natural_freq = 32.02681072 / damping_const
natural_freq_hz = natural_freq / math.pi
damping_const, natural_freq, natural_freq_hz

(0.9891164057402161, 32.37921293604708, 10.30662358439387)